# **OpenPyTEA** Walkthrough — Part 2: Creating the Plant

This notebook is **Part 2** of the OpenPyTEA walkthrough series:

1. [Part 1: Defining Equipment](part_1_equipment.ipynb)
2. **Part 2: Creating the Plant** (this notebook)
3. [Part 3: Cost Analysis and Sensitivity](part_3_analysis.ipynb)
4. [Part 4: Monte Carlo Uncertainty Analysis](part_4_monte_carlo.ipynb)
5. [Part 5: TEA Using Configuration Files](part_5_configuration_files.ipynb)

## ⚙️ Setup

This part uses the three `Equipment` objects (`hx`, `comp1`, `comp2`) that the demo plant is built from, which were built step by step in Part 1. The cell below recreates them so this notebook can run on its own (make sure **OpenPyTEA** is installed, e.g. `%pip install openpytea`).

In [1]:
from openpytea import Equipment

# The three pieces of equipment defined in Part 1 (usage examples 1, 2 and 4):
hx = Equipment(
    name="HX-101",
    param=900,                      # m² area
    process_type="Fluids",
    category="Heat Exchangers",
    type="U-tube shell & tube",
    material="316 stainless steel"
)

comp1 = Equipment(
    name='Comp-1',
    process_type='Fluids',
    material='Carbon steel',
    param=1,                        # MW
    category='Compressors, fans, & Blowers',
    type='Compressor, centrifugal',
    cost_func='co2_compressor_manzolini_2011'
)

comp2 = Equipment(
    name="Air Compressor",
    param=50_000,                   # exceeds upper_parallel -> auto-splits into parallel units
    process_type="Fluids",
    category='Compressors, fans, & blowers',
    type='Compressor, centrifugal',
)

## 🏭 Creating the Plant

`Plant` aggregates equipment and economics to estimate:
- **Fixed CAPEX** (ISBL / OSBL / D&E / Contingency) using process-type multipliers.
- **Location-adjusted costs** via country/region location factors.    
- **Variable OPEX** from itemized consumptions and prices (`variable_opex_inputs`). 
- **Fixed OPEX** (labor, supervision, overheads, maintenance, taxes/insurance, etc.), including automatic **working-capital** estimation when not supplied.
- **Operating labor costs** from an internal labor model that counts operators (when not provided) from the equipment composition (solids/fluids/mixed).  
- **Revenues** for a main product and optional by- or co-products.  
- **Cash flow, NPV, LCOP, payback, ROI, IRR** over the project lifetime, using your utilization, tax rate, depreciation model, and discount rate(s), with support for multi-scenario arrays.

Together, this makes `Plant` a self-contained TEA engine that turns your equipment list and economic assumptions into a full cash-flow and performance profile.

Let's now import the plant module and create a Plant object:

In [2]:
from openpytea import Plant

### Configurating a plant

In [3]:
config = {
    "plant_name": "Demo Plant",                 # The name of your plant. This will be used in plots
    # Basic plant information
    "process_type": "Fluids",                   # "Solids" | "Fluids" | "Mixed"
    "country": "United States",                 # Scroll down for country options. Optional, defaults to "United States"
    "region": "Gulf Coast",                     # Scroll down for region options. Optional, defaults to "Gulf Coast"
    "currency": "USD",                          # Currency code (e.g., "USD", "EUR"). Optional, defaults to "USD". 
                                                # Use \ when using symbol (e.g., "\$") to avoid syntax errors.
    "exchange_rate": 1.0,                       # Optional, defaults to 1.0. Use this if your costs are in a different currency than USD.
    "equipment": [hx, comp1, comp2],            # List of Equipment (each has .direct_cost)
    "interest_rate": 0.09,                      # Discount/interest rate. Optional, defaults to 0.09
    "project_lifetime": 30,                     # In years (needs to be int ≥ 3). Optional, defaults to 20
    "plant_utilization": 0.90,                  # 0-1. Optional, defaults to 1
    "tax_rate": 0.25,                           # 0-1, not used in levelized cost calculations. Optional, defaults to 0
    
    # Operator labor details
    'operator_hourly_rate': {
              'rate': 35,                       # This is in $/hour. Optional, defaults to  $38.11/hour 
              },
    'working_weeks_per_year': 46,               # This is the number of weeks the operator works. Optional, defaults to 49
    'working_shifts_per_week': 5,               # This is the number of shifts the operator works per week. Optional, defaults to 5

    # Plant products information
    'plant_products': {                         # Here we define the products produced by the plant
              'methanol': {                     # The first entry will be the main product. Important for levelized cost calculations.
                  'production': 125_000,        # Daily production in units/day
                  'price': 2.5                  # Price in USD/unit. No need to be specified for levelized cost calculations.
              },
              'hydrogen': {                     # Second and subsequent products are by- or co-products (if any)
                  'production': 100_000,        # Daily production in units/day
                  'price': 2.0                  # Price in USD/unit
              }
            },

    # Plant variable operating expenses, such as utilities and raw materials
    "variable_opex_inputs": {                   # Here, we define variable operating expenses that recur annually
        "electricity":   {"consumption": 2.2e6, # Daily consumption in units/day
                          "price": 0.08},       # Price in USD/unit
        "cooling_water": {"consumption": 1.6e6, "price": 0.0007},
    },

    # Additional CAPEX and OPEX information
    "working_capital": None,                    # in USD. Optional, defaults to 15% of fixed capital
    'additional_capex_cost': [500_000, 200_000],# List of additional CAPEX occuring during the project. For example, equipment replacements.
    'additional_capex_years': [8, 15],          # List of years in which additional CAPEX costs occur

    # Depreciation (optional; defaults to straight-line if omitted). Scroll down for more details.
    "depreciation": {
        "method": "macrs",            # "straight_line" | "declining_balance" | "macrs"
        "macrs_class": 7,             # 3,5,7,10,15,20 (half-year convention)
        "service_start_year": 2       # first operating year in your ramp (0/0/40%/80%/100%)
    }
}

demo_plant = Plant(config)

print(demo_plant)

ProcessPlant Configuration
----------------------------------------
Plant Name:                 Demo Plant
Process Type:               Fluids
Country / Region:           United States / Gulf Coast
Interest Rate:              0.09
Project Lifetime (years):   30
Plant Utilization:          0.9
Tax Rate:                   0.25
Working Capital:            None
Depreciation Settings:      {
    "method": "macrs",
    "macrs_class": 7,
    "service_start_year": 2
}

Operator Labor Inputs
  Hourly Rate:              {
    "rate": 35
}
  Operators per Shift:      None
  Operators Hired:          None
  Working Weeks / Year:     46
  Working Shifts / Week:    5
  Operating Shifts / Day:   3

Products
{
    "methanol": {
        "production": 125000,
        "price": 2.5
    },
    "hydrogen": {
        "production": 100000,
        "price": 2.0
    }
}

Variable OPEX Inputs:
{
    "electricity": {
        "consumption": 2200000.0,
        "price": 0.08
    },
    "cooling_water": {
        "con

Update any setting later without **rebuilding the plant** using:

In [4]:
config = {
    'project_lifetime': 20,
    'interest_rate': 0.08,
    'variable_opex_inputs': {
        'cooling_water': {"consumption": 0.8e6, "price": 0.0007},
        "steam": {"consumption": 4.0e5, "price": 0.02},
    }
}

demo_plant.update_configuration(config)
print(demo_plant)

ProcessPlant Configuration
----------------------------------------
Plant Name:                 Demo Plant
Process Type:               Fluids
Country / Region:           United States / Gulf Coast
Interest Rate:              0.08
Project Lifetime (years):   20
Plant Utilization:          0.9
Tax Rate:                   0.25
Working Capital:            None
Depreciation Settings:      {
    "method": "macrs",
    "macrs_class": 7,
    "service_start_year": 2
}

Operator Labor Inputs
  Hourly Rate:              {
    "rate": 35
}
  Operators per Shift:      None
  Operators Hired:          None
  Working Weeks / Year:     46
  Working Shifts / Week:    5
  Operating Shifts / Day:   3

Products
{
    "methanol": {
        "production": 125000,
        "price": 2.5
    },
    "hydrogen": {
        "production": 100000,
        "price": 2.0
    }
}

Variable OPEX Inputs:
{
    "electricity": {
        "consumption": 2200000.0,
        "price": 0.08
    },
    "cooling_water": {
        "con

Nested dictionaries (e.g., `variable_opex_inputs`, `plant_products`, `operator_hourly_rate`) are merged recursively, so unspecified sub-keys are preserved rather than overwritten.

### Summary of All Plant Configuration Keys

| **Key**                           | **Type / Allowed Values**                              | **Description**                                                                                                                 |
| --------------------------------- | ------------------------------------------------------ | ------------------------------------------------------------------------------------------------------------------------------- |
| `plant_name`                      | `str`                                                  | Name used in plots.                                                                                                             |
| `process_type`                    | `"Solids"`, `"Fluids"`, `"Mixed"`                      | Determines default cost factors for OSBL, D&E, and contingency.                                                                |
| `country`, `region`               | `str`                                                  | Used for location cost factors. `region` required when country has regional factors. Scroll down for options.                   |
| `loc_factor`                      | `float` or `None`                                      | Direct location factor applied to ISBL. Overrides `country`/`region` lookup when set. Default: `None`.                         |
| `currency`                        | `str`                                                  | Currency used for all economic inputs and outputs. Default: `"USD"`. Use `\` when using symbol (e.g., `"\$"`) to avoid syntax errors. |
| `exchange_rate`                   | `float`                                                | Conversion from USD to the selected `currency` (i.e., 1 USD = `exchange_rate` x `currency`). Required if `currency` is not USD. |
| `equipment`                       | `list[Equipment]`                                      | List of equipment objects. Each must expose `.direct_cost`, either computed or user-given.                                      |
| `interest_rate`                   | `float`                                                | Discount rate used for NPV, capital recovery, and working-capital interest. Default: 0.09.                                      |
| `project_lifetime`                | `int >= 3`                                             | Economic analysis horizon in years. Must be more than 3 years. Default: 20.                                                     |
| `plant_utilization`               | `0-1 float`                                            | Fraction of the year operating. Affects annual production and OPEX. Default: 1.                                                 |
| `tax_rate`                        | `0-1 float`                                            | Applied to net income for annual tax calculations (not used in levelized cost). Default: 0.                                     |
| **Operator labor settings**       |                                                        |                                                                                                                                 |
| `operator_hourly_rate`            | `{ "rate": float }`                                    | Operator labor cost model. If omitted, a default hourly rate is used.                                                           |
| `operators_per_shift`             | `int` or `None`                                        | If `None`, estimated from process steps/equipment.                                                                              |
| `operators_hired`                 | `int` or `None`                                        | If `None`, computed from `operators_per_shift` and shift schedule.                                                              |
| `production_type`                 | `"continuous"` or `"batch"`                            | Only affects the auto-calculated `operators_per_shift` (batch applies a minimum of 3 operators/shift). Does not affect capital, OPEX, or cash-flow calculations elsewhere.                            |
| `working_weeks_per_year`          | `int`                                                  | Used to convert hourly rate to annual labor cost. Default: 49.                                                                  |
| `working_shifts_per_week`         | `int`                                                  | Number of shifts per week the operators work. Default: 5.                                                                       |
| **Products and revenues**         |                                                        |                                                                                                                                 |
| `product_inputs`                  | `dict` of `{product: {production, price}}`             | First product = **main product** for levelized cost. `production` is **daily units**, automatically annualized via utilization. |
| **Variable operating expenses**   |                                                        |                                                                                                                                 |
| `variable_opex_inputs`            | `{item: {consumption, price}}`                         | Daily consumption and unit price. Annualized in OPEX calculations.                                                              |
| **Capital costs**                 |                                                        |                                                                                                                                 |
| `working_capital`                 | `float` or `None`                                      | If `None`, defaults to **15% of fixed capital**.                                                                                |
| `additional_capex_cost`           | `list[float]`                                          | Lump-sum CAPEX events (e.g., equipment replacements).                                                                           |
| `additional_capex_years`          | `list[int]`                                            | Year corresponding to each entry in `additional_capex_cost`.                                                                    |
| `fc`                              | `float` or `None`                                      | Multiplier on equipment **direct cost** to compute installed cost. Default: 1.                                                  |
| `fp`                              | `float` or `None`                                      | Multiplier on **fixed production costs**. Default: 1.                                                                           |
| **Capital cost factor overrides** |                                                        |                                                                                                                                 |
| `fixed_capital_factors`           | `dict`                                               | Override factor multipliers for individual capital components. Keys: `"osbl"`, `"de"`, `"contingency"`. Omitted or `None` values fall back to `process_type` defaults. Default: `{}`.                    |
| `fixed_capital_components`        | `dict`                                               | Override the absolute cost of individual capital components directly. Takes precedence over `fixed_capital_factors`. Keys: `"osbl"`, `"dne"`, `"contingency"`. Default: `{}`.                              |
| **Cash flow profiles**            |                                                        |                                                                                                                                 |
| `capex_ramp`                      | `list[float]` or `None`                                | Fraction of fixed capital spent in each construction year. Must be non-negative and **sum to 1.0**. Length must be less than `project_lifetime`. Working capital is drawn in the final construction year. Default: `[0.3, 0.6, 0.1]`. |
| `production_ramp`                 | `list[float]` or `None`                                | Nameplate capacity utilisation per project year (values 0-1). Years beyond the list are set to 1.0. Length must not exceed `project_lifetime`. Default: `[0, 0, 0.4, 0.8]`. |
| **Fixed OPEX customisation**      |                                                        |                                                                                                                                 |
| `fixed_opex_factors`              | `dict` or `{}`                                         | Override the multipliers for individual fixed OPEX components. Any subset of keys may be supplied; omitted keys use defaults. Keys: `"supervision"` (0.25), `"direct_salary_overhead"` (0.5), `"laboratory_charges"` (0.10), `"maintenance"` (0.05), `"taxes_insurance"` (0.015), `"rent_of_land"` (0.015), `"environmental_charges"` (0.01), `"operating_supplies"` (0.009), `"general_plant_overhead"` (0.65), `"working_capital"` (0.15), `"patents_royalties"` (0.02), `"distribution_selling"` (0.02), `"rnd"` (0.03). |
| `fixed_opex_components`           | `dict` or `{}`                                         | Override the computed **cost value** of individual fixed OPEX components directly, bypassing the factor-based formula. Downstream components that depend on an overridden value use the overridden value. Keys: `"supervision_costs"`, `"direct_salary_overhead"`, `"laboratory_charges"`, `"maintenance_costs"`, `"taxes_insurance_costs"`, `"rent_of_land_costs"`, `"environmental_charges"`, `"operating_supplies"`, `"general_plant_overhead"`, `"patents_royalties"`, `"distribution_selling_costs"`, `"RnD_costs"`. |
| **Depreciation settings**         |                                                        |                                                                                                                                 |
| `depreciation`                    | `dict` or `None`                                       | Optional depreciation configuration block. If omitted, default **straight-line** depreciation is applied.                      |
| `depreciation.method`             | `"straight_line"` / `"declining_balance"` / `"macrs"` | Depreciation model to use. Defaults to `"straight_line"` if not provided.                                                      |
| `depreciation.life`               | `int`                                                  | Asset service life in years. Used for straight-line and declining-balance methods.                                              |
| `depreciation.salvage_fraction`   | `float` (0-1)                                          | Fraction of value remaining at end of life (not depreciated). Default: `0.0`. Applies to straight-line and declining-balance.   |
| `depreciation.db_factor`          | `float`                                                | Declining-balance factor, e.g., `2.0` for 200% DDB or `1.5` for 150% DB. Auto-switches to straight-line when advantageous.    |
| `depreciation.macrs_class`        | `int` in {3, 5, 7, 10, 15, 20}                         | MACRS recovery period (years). Determines the predefined IRS depreciation schedule (half-year convention).                     |
| `depreciation.convention`         | `"half_year"` *(default)*                              | Depreciation timing convention for MACRS. Only `"half_year"` is currently implemented.                                         |
| `depreciation.service_start_year` | `int >= 0`                                             | Year (relative to start-up) when depreciation begins. Common choice: **2** (first full operating year). Applies to all methods. |

### Methodology and Calculations

#### **1. Fixed Capital Investment**

The total fixed capital investment (FCI) of the plant consists of:
- Inside Battery Limits (ISBL)  
- Outside Battery Limits (OSBL)  
- Design and engineering costs (D&E)
- Contingency

Estimated using the following equation:
$$
\text{FCI} = \text{ISBL} \cdot (1 + OS) \cdot (1 + D\&E + X) \cdot LF
$$

*Source: Towler, G.; Sinnott, R. Chemical Engineering Design; Elsevier, 2022. https://doi.org/10.1016/C2019-0-02025-0*

<p style="text-align: justify;">Where ISBL is the total of equipment direct costs, and the remaining factors are determined by the plant configuration. By default they are looked up from <code>process_type</code>, <code>country</code>, and <code>region</code>. Each factor can also be overridden directly in the plant config.</p>

- **Process type**: Determines the nature of the plant operation and sets the default cost factors. The available types and their defaults are:
    - `Solids`: OS = 0.4, D\&E = 0.2, X = 0.1  
    - `Fluids`: OS = 0.3, D\&E = 0.3, X = 0.1  
    - `Mixed`: OS = 0.4, D\&E = 0.25, X = 0.1  

*Source: Towler, G.; Sinnott, R. Chemical Engineering Design; Elsevier, 2022. https://doi.org/10.1016/C2019-0-02025-0*

  > **Overriding individual factors**: use `fixed_capital_factors` to replace specific process-type defaults, or `fixed_capital_components` to set absolute cost values directly.
  > ```python
  > plant_config = {
  >     "process_type": "Fluids",  # sets defaults: OS=0.3, D&E=0.3, X=0.1
  >     "fixed_capital_factors": {
  >         "osbl": 0.25,        # overrides OS  -> 0.25 (default 0.3)
  >         "contingency": 0.15  # overrides X   -> 0.15 (default 0.1)
  >         # "de" not set      -> D&E stays 0.3
  >     },
  >     # or set absolute costs directly (takes precedence over factors):
  >     # "fixed_capital_components": {"osbl": 5_000_000}
  > }
  > ```

- **Country & region**: Determines the location factor (_LF_), which adjusts the estimated capital cost to reflect local economic conditions and construction costs. Available countries and regions are:  

    - **United States**
        - Gulf Coast: 1.00  
        - East Coast: 1.04  
        - West Coast: 1.07  
        - Midwest: 1.02  
    - **Canada** 
        - Ontario: 1.00  
        - Fort McMurray: 1.60  
    - **Mexico**: 1.03  
    - **Brazil**: 1.14  
    - **China**  
        - Imported: 1.12  
        - Indigenous: 0.61  
    - **Japan**: 1.26  
    - **Southeast Asia**: 1.12  
    - **Australia**: 1.21  
    - **India**: 1.02  
    - **Middle East**: 1.07  
    - **France**: 1.13  
    - **Germany**: 1.11  
    - **Italy**: 1.14  
    - **Netherlands**: 1.19  
    - **Russia**: 1.53  
    - **United Kingdom**: 1.02  

*Source: Towler, G.; Sinnott, R. Chemical Engineering Design; Elsevier, 2022. https://doi.org/10.1016/C2019-0-02025-0*

  > **Overriding the location factor**: set `loc_factor` in the plant config to bypass the country/region lookup entirely.
  > ```python
  > plant_config = {
  >     "loc_factor": 1.15  # skips country/region lookup
  > }
  > ```

In [5]:
demo_plant.calculate_fixed_capital(print_results=True)


Capital cost estimation
ISBL: 98,336,036.03 USD
OSBL: 29,500,810.81 USD
Design and engineering: 38,351,054.05 USD
Contingency: 12,783,684.68 USD
Fixed capital investment: 178,971,585.57 USD


We can override any of the default factors via update_configuration and recalculate:

| Override key | Sub-key | Replaces | Demo value | Default (Fluids) |
|---|---|---|---|---|
| `fixed_capital_factors` | `"osbl"` | OS | 0.25 | 0.30 |
| `fixed_capital_factors` | `"de"` | D&E | 0.35 | 0.30 |
| `fixed_capital_factors` | `"contingency"` | X | 0.15 | 0.10 |
| `loc_factor` | — | LF | 1.10 | 1.19 (Netherlands) |


In [6]:
# Override the default process-type and location factors
demo_plant.update_configuration({
    "fixed_capital_factors": {
        "osbl": 0.25,        # default for Fluids: 0.30
        "de": 0.35,          # default for Fluids: 0.30
    },
    "fixed_capital_components": {
        "contingency": 15_000_000 # USD, default for Fluids: 0.10
    },
    "loc_factor": 1.10,         # default for Netherlands: 1.19
})

demo_plant.calculate_fixed_capital(print_results=True)

Capital cost estimation
ISBL: 108,169,639.63 USD
OSBL: 27,042,409.91 USD
Design and engineering: 47,324,217.34 USD
Contingency: 15,000,000.00 USD
Fixed capital investment: 197,536,266.88 USD


If you need only the ISBL, you can just perform `calculate_isbl`:

In [7]:
demo_plant.calculate_isbl(print_results=True)

ISBL cost estimation
  - HX-101: 1,181,186.56 USD
  - Comp-1: 10,840,781.13 USD
  - Air Compressor: 86,314,068.34 USD
Total ISBL: 108,169,639.63 USD


To retrieve the values of cost estimates for each component of capital cost, you can simply access the corresponding attributes:

In [8]:
print(f"OSBL: ${demo_plant.osbl:,.2f}")
print(f'Design and Engineering: ${demo_plant.dne:,.2f}')
#  ... and so on

OSBL: $27,042,409.91
Design and Engineering: $47,324,217.34


#### **2. Fixed Operating Expenditure**

<p style="text-align: justify;">Fixed production costs are those costs that do not vary with the rate of production. They are calculated using the <code>calculate_fixed_opex</code> method of the <code>Plant</code> object. 

---

##### Cost Components and Default Formulas

The table below lists every component, the basis it is multiplied against, and the default factor. All factors can be customised (see below).

| Component | Default formula | `fixed_opex_factors` key |
|---|---|---|
| Operating labor | Computed from shift schedule and operator hourly rate | — |
| Supervision | 0.25 × operating labor | `"supervision"` |
| Direct salary overhead | 0.50 × (operating labor + supervision) | `"direct_salary_overhead"` |
| Laboratory charges | 0.10 × operating labor | `"laboratory_charges"` |
| Maintenance | 0.05 × ISBL | `"maintenance"` |
| Taxes & insurance | 0.015 × ISBL | `"taxes_insurance"` |
| Rent of land | 0.015 × (ISBL + OSBL) | `"rent_of_land"` |
| Environmental charges | 0.01 × (ISBL + OSBL) | `"environmental_charges"` |
| Operating supplies | 0.009 × ISBL | `"operating_supplies"` |
| General plant overhead | 0.65 × (operating labor + supervision + direct salary overhead) | `"general_plant_overhead"` |
| Interest on working capital | working capital × interest rate | `"working_capital"` (default 0.15 × FCI) |
| Patents & royalties | 0.02 × cash cost of production* | `"patents_royalties"` |
| Distribution & selling | 0.02 × cash cost of production* | `"distribution_selling"` |
| R&D | 0.03 × cash cost of production* | `"rnd"` |

*Source: Turton, R.; Shaeiwitz, J. A.; Bhattacharyya, D.; Whiting, W. B. Analysis, Synthesis, and Design of Chemical Processes, 5th ed.; Prentice Hall, 2018.*


<p style="text-align: justify;">* Cash cost of production = (variable costs + fixed costs so far) / (1 − sum of the three rates above). This ensures patents, selling, and R&D are expressed as a consistent fraction of total cash cost, so changing any one of their rates automatically adjusts the denominator.</p>

The working capital used to compute interest defaults to 15% of fixed capital when not explicitly set via the `"working_capital"` plant config key.

---

> **Overriding Estimation Factors**
>
> Use `fixed_opex_factors` in the plant config (or via `update_configuration`) to replace the default multiplier for any component. Only the keys you supply are changed — all others keep their defaults.
>
> ```python
> demo_plant.update_configuration({
>     "fixed_opex_factors": {
>         "maintenance": 0.06,       # 6% of ISBL instead of 5%
>         "rnd": 0.0,                # zero out R&D
>         "patents_royalties": 0.03, # 3% of cash cost instead of 2%
>     }
> })
> demo_plant.calculate_fixed_opex(print_results=True)
> ```

> **Overriding Component Values Directly**
>
> Use `fixed_opex_components` to supply an absolute cost value for a component, bypassing the factor formula entirely. Components that depend on an overridden value (e.g. `direct_salary_overhead` depends on `supervision_costs`) automatically use the overridden value in their own calculation.
>
> ```python
> demo_plant.update_configuration({
>     "fixed_opex_components": {
>         "supervision_costs": 80_000,   # fixed value, ignores the 0.25 factor
>         "maintenance_costs": 150_000,  # fixed value, ignores the 0.05 factor
>     }
> })
> demo_plant.calculate_fixed_opex(print_results=True)
> ```
>
> Available keys: `"supervision_costs"`, `"direct_salary_overhead"`, `"laboratory_charges"`, `"maintenance_costs"`, `"taxes_insurance_costs"`, `"rent_of_land_costs"`, `"environmental_charges"`, `"operating_supplies"`, `"general_plant_overhead"`, `"patents_royalties"`, `"distribution_selling_costs"`, `"RnD_costs"`
>
> **Precedence**: `fixed_opex_components` takes priority over `fixed_opex_factors` for the same component.

---

##### Operating labor

Operating labor cost is calculated as:

$$
C_{\text{labor}} = N_{\text{hired}} \times H_{\text{year}} \times r
$$

where:
- $N_{\text{hired}}$ — number of operators hired (see below)
- $H_{\text{year}} = W_{\text{weeks}} \times W_{\text{shifts}} \times (24 / S_{\text{day}})$ — working hours per operator per year
- $r$ — operator hourly rate (default: **\$38.11/hr**)

**Number of operators per shift** is estimated from the equipment list. For processes with at most 2 solids-handling sections ($N_{\text{solid}} \le 2$), the Turton et al. empirical correlation applies:

$$
N_{\text{shift}} = \sqrt{6.29 + 31.7 \cdot N_{\text{solid}}^2 + 0.23 \cdot N_{\text{fluid}}}
$$

*Source: Turton, R.; Shaeiwitz, J. A.; Bhattacharyya, D.; Whiting, W. B. Analysis, Synthesis, and Design of Chemical Processes, 5th ed.; Prentice Hall, 2018.*

Beyond that range ($N_{\text{solid}} > 2$), the correlation is no longer valid, and the package falls back to the rule-based chart method instead:

$$
N_{\text{shift}} = 3 + N_{\text{solid}}
$$

*Source: Towler, G.; Sinnott, R. Chemical Engineering Design; Elsevier, 2022 (Figure 8.12). https://doi.org/10.1016/C2019-0-02025-0**

where $N_{\text{solid}}$ and $N_{\text{fluid}}$ are the numbers of solid-handling and fluid-handling process steps (pumps and pressure vessels excluded), respectively.

For **batch processes** (`production_type: "batch"` in the plant config), the chart gives no formula for staffing — only a minimum of 3 operators per shift, properly determined from the batch sequence and degree of automation. OpenPyTEA applies that minimum as a floor on top of whichever estimate above applies:

$$
N_{\text{shift}} \leftarrow \max(3, N_{\text{shift}})
$$

`production_type` defaults to `"continuous"` and only affects this staffing estimate — it does not change capital, OPEX, or cash-flow calculations elsewhere.

**Total operators hired** accounts for the gap between the plant's continuous operating schedule and each operator's working schedule:

$$
N_{\text{hired}} = \left\lceil N_{\text{shift}} \times \frac{365 \times S_{\text{day}}}{W_{\text{weeks}} \times W_{\text{shifts}}} \right\rceil
$$

*Source: Turton, R.; Shaeiwitz, J. A.; Bhattacharyya, D.; Whiting, W. B. Analysis, Synthesis, and Design of Chemical Processes, 5th ed.; Prentice Hall, 2018.*

The schedule parameters ($W_{\text{weeks}}$, $W_{\text{shifts}}$, $S_{\text{day}}$) are set via `working_weeks_per_year` (default: 49), `working_shifts_per_week` (default: 5), and `operating_shifts_per_day` (default: 3) in the plant config.

The operator hourly rate is specified as:

```python
"operator_hourly_rate": {
    "rate": <hourly_rate>   # default: 38.11
}
```

> **Overriding operator labor**
>
> Any part of the labor calculation can be bypassed:
>
> | Config key | Effect |
> |---|---|
> | `operator_hourly_rate: {"rate": r}` | Sets the hourly wage rate |
> | `operators_per_shift` | Skips the empirical formula; uses this value directly |
> | `operators_hired` | Skips both the formula and the hired calculation; uses this value directly |
> | `working_weeks_per_year`, `working_shifts_per_week`, `operating_shifts_per_day` | Adjust the shift schedule used for both hired count and annual hours |
>
> ```python
> demo_plant.update_configuration({
>     "operators_per_shift": 5,              # bypass empirical formula
>     "operator_hourly_rate": {"rate": 45.0} # custom wage
> })
> demo_plant.calculate_fixed_opex(print_results=True)
> ```

Call to calculate Fixed OPEX:


In [9]:
demo_plant.calculate_fixed_opex(print_results=True)  # Calculate fixed OPEX costs

Fixed production costs estimation
Operating labor costs: 837,200.00 USD per year
Supervision costs: 209,300.00 USD per year
Direct salary overhead: 523,250.00 USD per year
Laboratory charges: 83,720.00 USD per year
Maintenance costs: 5,408,481.98 USD per year
Taxes and insurance costs: 1,622,544.59 USD per year
Rent of land costs: 2,028,180.74 USD per year
Environmental charges: 1,352,120.50 USD per year
Operating supplies: 973,526.76 USD per year
General plant overhead: 1,020,337.50 USD per year
Interest on working capital: 2,370,435.20 USD per year
Patents and royalties: 1,657,141.02 USD per year
Distribution and selling costs: 1,657,141.02 USD per year
R&D costs: 2,485,711.52 USD per year
Fixed OPEX: 22,229,090.83 USD per year


---
##### Modifying Labor Costs

Override any of the labor parameters below using `update_configuration`:

In [10]:
# --- Example: modify labor staffing and wages ---

# By default operators_hired and operators_per_shift are auto-calculated
# from the process type. Print the auto-calculated values first:
print(f"Auto operators per shift : {demo_plant.calculate_operators_per_shift():.1f}")
print(f"Auto operators hired     : {demo_plant.calculate_operators_hired()}")
print()

# Now override the labor parameters:
demo_plant.update_configuration({
    # Pin headcount manually (set to None to revert to auto-calculation)
    "operators_hired": 12,

    # Adjust the working schedule
    "working_weeks_per_year": 46,   # default: 49
    "working_shifts_per_week": 5,   # default: 5
    "operating_shifts_per_day": 3,  # default: 3 (how many shifts the plant runs/day)

    # Raise the hourly wage
    "operator_hourly_rate": {"rate": 42.0},  # default: ~38.11 $/hr
})

demo_plant.calculate_fixed_opex()
print(f"Operators hired          : {demo_plant.calculate_operators_hired()}")
print(f"Operating labor costs    : {demo_plant.operating_labor_costs:,.2f} {demo_plant.currency}/yr")

Auto operators per shift : 2.6
Auto operators hired     : 13

Operators hired          : 12
Operating labor costs    : 927,360.00 USD/yr


---
##### Modifying Fixed OPEX Factors and Costs
We can adjust the fixed OPEX parameters in two ways:

1. **`fixed_opex_factors`** — override the multiplier for a component (formula still runs, just with a different factor)
2. **`fixed_opex_components`** — supply an absolute cost value, bypassing the formula entirely

The example below overrides three factors and pins one component value directly, then recalculates.

In [11]:
# --- Override estimation factors ---
# Change maintenance from 5% to 6% of ISBL,
# set rent of land to 1% of (ISBL+OSBL) instead of 1.5%,
# and zero out R&D costs.
demo_plant.update_configuration({
    "fixed_opex_factors": {
        "maintenance": 0.06,    # default: 0.05 x ISBL
        "rent_of_land": 0.01,   # default: 0.015 x (ISBL + OSBL)
        "rnd": 0.0,             # default: 0.03 x cash cost of production
    }
})

# --- Override a component value directly ---
# Set supervision costs to a fixed value instead of 25% of operating labor.
demo_plant.update_configuration({
    "fixed_opex_components": {
        "supervision_costs": 100_000,  # default: 0.25 x operating_labor_costs
    }
})

demo_plant.calculate_fixed_opex(print_results=True)

Fixed production costs estimation
Operating labor costs: 927,360.00 USD per year
Supervision costs: 100,000.00 USD per year
Direct salary overhead: 513,680.00 USD per year
Laboratory charges: 92,736.00 USD per year
Maintenance costs: 6,490,178.38 USD per year
Taxes and insurance costs: 1,622,544.59 USD per year
Rent of land costs: 1,352,120.50 USD per year
Environmental charges: 1,352,120.50 USD per year
Operating supplies: 973,526.76 USD per year
General plant overhead: 1,001,676.00 USD per year
Interest on working capital: 2,370,435.20 USD per year
Patents and royalties: 1,613,007.04 USD per year
Distribution and selling costs: 1,613,007.04 USD per year
R&D costs: 0.00 USD per year
Fixed OPEX: 20,022,392.00 USD per year


To retrieve the values for each fixed operating cost components, you can do:

In [12]:
print(f'Operating labor costs: ${demo_plant.operating_labor_costs:,.2f}')
print(f'Supervision costs: ${demo_plant.supervision_costs:,.2f}')
print(f'Maintenance costs: ${demo_plant.maintenance_costs:,.2f}')
# ..... and so on

Operating labor costs: $927,360.00
Supervision costs: $100,000.00
Maintenance costs: $6,490,178.38


#### **3. Variable Operating Expenditure**

Variable operating expenses represent consumables and utilities that scale directly with plant production.
These are defined through the `variable_opex_inputs` dictionary in your plant configuration and are automatically processed inside `calculate_variable_opex()`.

Each entry in `variable_opex_inputs` follows the format:
```python
"variable_opex_inputs": {
    "<item_name>": {
        "consumption": <annual_quantity>,
        "price": <unit_price>
    },
    ...
}
```

During calculation:
1. Each item’s annual cost is computed as:

            cost = daily_consumption × price × 365 x plant_utilization

2. All items are summed to form the total variable operating cost (`self.variable_production_costs`).

3. The results are stored in the plant instance and can be printed when `print_results=True.`

For example:

In [13]:
config = {
    "variable_opex_inputs": {
        "electricity":   {"consumption": 1.4e6, "price": 0.075},
        "cooling_water": {"consumption": 1.6e6, "price": 0.0007},
        "steam":         {"consumption": 4.0e5, "price": 0.02},
        "natural_gas":   {"consumption": 1.0e5, "price": 0.035}
    },
}

demo_plant.update_configuration(config)

Then run:

In [14]:
demo_plant.calculate_variable_opex(print_results=True)

Variable production costs estimation
  - Electricity: 34,492,500.00 USD per year
  - Cooling water: 367,920.00 USD per year
  - Steam: 2,628,000.00 USD per year
  - Natural gas: 1,149,750.00 USD per year
Total Variable OPEX: 38,638,170.00 USD per year


#### **4. Revenue Calculation**

Plant revenue is computed from the products defined in the `plant_products` dictionary inside your configuration.
The first product listed is treated as the main product for levelized cost calculations, but all products contribute to total annual revenue.

Each product entry follows this structure:
```python
"plant_products": {
    "<main_product_name>": {
        "production": <daily_production_rate>,
        "price": <unit_price>
    },
    "<side_product_name>": {
        "production": <daily_production_rate>,
        "price": <unit_price>
    },
    ...
}
```

During processing, each product's annual revenue is computed as:

            annual_revenue = daily_production × price × 365 × plant_utilization
Then:
1. Each product’s annual revenue is calculated individually.
2. All product revenues are summed to produce the plant’s total annual revenue.
3. Results are stored on the plant instance (e.g., self.revenue_breakdown, self.total_revenue) and appear in output tables when print_results=True.

For example:

In [15]:
config = {
    "plant_products": {
        "methanol": {"production": 100_000, "price": 1.95},
        "hydrogen": {"production": 75_000, "price": 1.25}
    },
}

demo_plant.update_configuration(config)

Then you can run:

In [16]:
demo_plant.calculate_revenue(print_results=True)

Revenue estimation
  - Methanol: 64,057,500.00 USD per year
  - Hydrogen: 30,796,875.00 USD per year
Total Revenue: 94,854,375.00 USD per year


#### **5. Cash Flow Calculation**

The cash flow represents the net annual financial performance of the plant, combining revenues, operating costs, capital investments, taxes, and depreciation.
It is generated by calling:

In [17]:
demo_plant.calculate_cash_flow(print_results=True)

,Year,Capital cost [USD],Revenue [USD],Cash cost [USD],Gross profit [USD],Depreciation [USD],Taxable income [USD],Tax paid [USD],Cash flow [USD]
0,1,"59,260,880.06",0.00,"19,106,150.75","-19,106,150.75",0.00,"-19,106,150.75",0.00,"-78,367,030.82"
1,2,"118,521,760.13",0.00,"19,106,150.75","-19,106,150.75",0.00,"-19,106,150.75",0.00,"-137,627,910.88"
2,3,"49,384,066.72","37,941,750.00","34,561,418.75","3,380,331.25","28,227,932.54","-24,847,601.29",0.00,"-46,003,735.47"
3,4,0.00,"75,883,500.00","50,016,686.75","25,866,813.25","48,376,631.76","-22,509,818.51",0.00,"25,866,813.25"
4,5,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","34,549,093.08","2,560,961.17",0.00,"37,110,054.25"
5,6,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","24,672,279.73","12,437,774.51","640,240.29","36,469,813.95"
6,7,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","17,639,988.63","19,470,065.62","3,109,443.63","34,000,610.62"
7,8,"500,000.00","94,854,375.00","57,744,320.75","37,110,054.25","17,620,235.01","19,489,819.24","4,867,516.40","31,742,537.84"
8,9,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","17,639,988.63","19,470,065.62","4,872,454.81","32,237,599.44"
9,10,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","8,810,117.50","28,299,936.74","4,867,516.40","32,242,537.84"


The method builds yearly arrays for capital cost, production, revenue, OPEX, depreciation, taxable income, and cash flow over the defined project lifetime.

---
**Capital Expenditure (CAPEX) Ramp**

Capital spending is spread across construction years as a fraction of the total fixed capital investment. The default profile spans three years:

| **Year**   | **CAPEX Fraction** | **Description**                              |
| ---------- | ------------------ | -------------------------------------------- |
| 0          | 30%                | Initial design and early construction        |
| 1          | 60%                | Major equipment procurement and installation |
| 2          | 10%                | Commissioning and start-up costs             |
| Final year | —                  | Working capital released (negative CAPEX)    |

Working capital is drawn in the final construction year and returned in the last project year.

> **Overriding the CAPEX ramp**
>
> Set `capex_ramp` in the plant config to a list of fractions that **sum to 1.0**. Each entry represents the fraction of fixed capital spent in that construction year. The length of the list defines the number of construction years; it must be shorter than `project_lifetime`. Working capital is automatically drawn in the last construction year.
>
> ```python
> demo_plant.update_configuration({
>     "capex_ramp": [0.2, 0.5, 0.2, 0.1]  # 4-year build instead of 3
> })
> demo_plant.calculate_cash_flow(print_results=True)
> ```
>
> **Validation**: all values must be ≥ 0, the list must sum to 1.0 (tolerance 1e-6), and its length must be less than `project_lifetime`.

---
**Production Ramp**

Plant output increases gradually from zero to full capacity. The default profile is:

| **Year** | **Production Level** |
| -------- | -------------------- |
| 0        | 0%                   |
| 1        | 0%                   |
| 2        | 40%                  |
| 3        | 80%                  |
| 4+       | 100% (steady state)  |

Annual production for each year is calculated as:

$$
\text{Annual production} = \text{daily\_prod} \times 365 \times \text{plant\_utilization} \times \text{ramp\_factor}
$$

Revenue and variable operating costs for each year also scale with the production ramp.

> **Overriding the production ramp**
>
> Set `production_ramp` to a list of capacity fractions (values 0–1), one per project year. Years beyond the list are automatically set to 1.0 (full capacity). The list must not be longer than `project_lifetime`.
>
> ```python
> demo_plant.update_configuration({
>     "production_ramp": [0, 0, 0, 0.3, 0.6, 0.9]  # slower 6-year ramp-up
> })
> demo_plant.calculate_cash_flow(print_results=True)
> ```
>
> **Validation**: all values must be between 0 and 1, and the list length must not exceed `project_lifetime`.


The example below modifies both ramps and recalculates the cash flow to show the effect.

In [18]:
# Override the CAPEX ramp: 4-year build (20/50/20/10)
# Override the production ramp: slower ramp-up over 6 years
demo_plant.update_configuration({
    "capex_ramp": [0.2, 0.5, 0.2, 0.1],          # 4 construction years, must sum to 1.0
    "production_ramp": [0, 0, 0, 0, 0.4, 0.8],   # years 0-3: construction, 4: 40%, 5: 80%, 6+: 100%
})

demo_plant.calculate_cash_flow(print_results=True)

,Year,Capital cost [USD],Revenue [USD],Cash cost [USD],Gross profit [USD],Depreciation [USD],Taxable income [USD],Tax paid [USD],Cash flow [USD]
0,1,"39,507,253.38",0.00,"19,106,150.75","-19,106,150.75",0.00,"-19,106,150.75",0.00,"-58,613,404.13"
1,2,"98,768,133.44",0.00,"19,106,150.75","-19,106,150.75",0.00,"-19,106,150.75",0.00,"-117,874,284.19"
2,3,"39,507,253.38",0.00,"19,106,150.75","-19,106,150.75","25,405,139.28","-44,511,290.04",0.00,"-58,613,404.13"
3,4,"49,384,066.72",0.00,"19,106,150.75","-19,106,150.75","46,361,761.84","-65,467,912.59",0.00,"-68,490,217.47"
4,5,0.00,"37,941,750.00","34,561,418.75","3,380,331.25","35,931,846.95","-32,551,515.70",0.00,"3,380,331.25"
5,6,0.00,"75,883,500.00","50,016,686.75","25,866,813.25","25,659,961.07","206,852.18",0.00,"25,866,813.25"
6,7,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","18,343,217.74","18,766,836.50","51,713.04","37,058,341.20"
7,8,"500,000.00","94,854,375.00","57,744,320.75","37,110,054.25","17,622,210.37","19,487,843.88","4,691,709.13","31,918,345.12"
8,9,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","17,638,013.27","19,472,040.98","4,871,960.97","32,238,093.28"
9,10,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","9,693,104.62","27,416,949.63","4,868,010.24","32,242,044.00"


---
**Depreciation**

Depreciation is applied using the selected depreciation method
(`straight_line`, `declining_balance`, or `macrs`) and begins in the configured `service_start_year`.

You can change the depreciation method as follows: 

In [19]:
# Switching to straight-line depreciation
demo_plant.update_configuration({
    "depreciation": {
        "method": "straight_line",
        "life": 12,
        "salvage_fraction": 0.05,
        "service_start_year": 2
    }
})

demo_plant.calculate_cash_flow(print_results=True)

,Year,Capital cost [USD],Revenue [USD],Cash cost [USD],Gross profit [USD],Depreciation [USD],Taxable income [USD],Tax paid [USD],Cash flow [USD]
0,1,"39,507,253.38",0.00,"19,106,150.75","-19,106,150.75",0.00,"-19,106,150.75",0.00,"-58,613,404.13"
1,2,"98,768,133.44",0.00,"19,106,150.75","-19,106,150.75",0.00,"-19,106,150.75",0.00,"-117,874,284.19"
2,3,"39,507,253.38",0.00,"19,106,150.75","-19,106,150.75","14,074,459.02","-33,180,609.77",0.00,"-58,613,404.13"
3,4,"49,384,066.72",0.00,"19,106,150.75","-19,106,150.75","15,638,287.79","-34,744,438.55",0.00,"-68,490,217.47"
4,5,0.00,"37,941,750.00","34,561,418.75","3,380,331.25","15,638,287.79","-12,257,956.55",0.00,"3,380,331.25"
5,6,0.00,"75,883,500.00","50,016,686.75","25,866,813.25","15,638,287.79","10,228,525.45",0.00,"25,866,813.25"
6,7,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","15,638,287.79","21,471,766.45","2,557,131.36","34,552,922.88"
7,8,"500,000.00","94,854,375.00","57,744,320.75","37,110,054.25","15,638,287.79","21,471,766.45","5,367,941.61","31,242,112.63"
8,9,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","15,638,287.79","21,471,766.45","5,367,941.61","31,742,112.63"
9,10,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","15,638,287.79","21,471,766.45","5,367,941.61","31,742,112.63"


In [20]:
# Switching to Declining-Balance (200% DDB)
demo_plant.update_configuration({
    "depreciation": {
        "method": "declining_balance",
        "life": 10,
        "db_factor": 2.0,        # 200% DDB (use 1.5 for 150% DB)
        "salvage_fraction": 0.1,
        "service_start_year": 2
    }
})

demo_plant.calculate_cash_flow(print_results=True)

,Year,Capital cost [USD],Revenue [USD],Cash cost [USD],Gross profit [USD],Depreciation [USD],Taxable income [USD],Tax paid [USD],Cash flow [USD]
0,1,"39,507,253.38",0.00,"19,106,150.75","-19,106,150.75",0.00,"-19,106,150.75",0.00,"-58,613,404.13"
1,2,"98,768,133.44",0.00,"19,106,150.75","-19,106,150.75",0.00,"-19,106,150.75",0.00,"-117,874,284.19"
2,3,"39,507,253.38",0.00,"19,106,150.75","-19,106,150.75","35,556,528.04","-54,662,678.79",0.00,"-58,613,404.13"
3,4,"49,384,066.72",0.00,"19,106,150.75","-19,106,150.75","32,395,947.77","-51,502,098.52",0.00,"-68,490,217.47"
4,5,0.00,"37,941,750.00","34,561,418.75","3,380,331.25","25,916,758.21","-22,536,426.97",0.00,"3,380,331.25"
5,6,0.00,"75,883,500.00","50,016,686.75","25,866,813.25","20,733,406.57","5,133,406.68",0.00,"25,866,813.25"
6,7,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","16,586,725.26","20,523,328.99","1,283,351.67","35,826,702.58"
7,8,"500,000.00","94,854,375.00","57,744,320.75","37,110,054.25","13,269,380.21","23,840,674.04","5,130,832.25","31,479,222.00"
8,9,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","10,615,504.16","26,494,550.08","5,960,168.51","31,149,885.74"
9,10,0.00,"94,854,375.00","57,744,320.75","37,110,054.25","8,492,403.33","28,617,650.92","6,623,637.52","30,486,416.73"


---

**Tax**

Taxable income is computed as:

            Taxable Income = Gross Profit − Depreciation


A *one-year lag* is applied to taxation:

taxes in year t are based on taxable income from year t−1.

---

**Cash Flow Formula**

The annual cash flow for each year is calculated as:

        Cash Flow = Gross Profit − Tax Paid − Capital Cost

where:
- Gross Profit = Revenue − (Fixed OPEX + Variable OPEX)
- Tax Paid = tax_rate × previous_year_taxable_income
- Capital Cost includes CAPEX spending and working capital adjustment

Now we can use the `calculate_variable_opex` function:

To retrieve the values for each variable operating cost components, you can do:

#### **6. Economic Performance Metrics**

After building cash flows, use these methods to evaluate project economics.

Run them after `calculate_cash_flow()` so the yearly arrays are available.

---

**Net Present Value (NPV)**

Concept: Present value of all yearly cash flows discounted at interest_rate.

Call:

In [21]:
demo_plant.calculate_npv(print_results=True)

Year | Present Value [USD] | Cumulative NPV [USD]
-------------------------------------------
   1 |  -54,271,670.49 |  -54,271,670.49
   2 | -101,058,199.75 | -155,329,870.24
   3 |  -46,529,209.95 | -201,859,080.20
   4 |  -50,342,354.47 | -252,201,434.66
   5 |    2,300,596.65 | -249,900,838.02
   6 |   16,300,480.05 | -233,600,357.96
   7 |   20,904,536.85 | -212,695,821.11
   8 |   17,007,244.15 | -195,688,576.96
   9 |   15,582,698.17 | -180,105,878.79
  10 |   14,121,109.70 | -165,984,769.09
  11 |   12,847,461.19 | -153,137,307.90
  12 |   11,733,028.47 | -141,404,279.44
  13 |   10,849,285.08 | -130,554,994.35
  14 |    9,532,869.64 | -121,022,124.71
  15 |    8,710,929.24 | -112,311,195.47
  16 |    8,124,053.31 | -104,187,142.16
  17 |    7,522,271.59 |  -96,664,870.57
  18 |    6,965,066.28 |  -89,699,804.29
  19 |    6,449,135.45 |  -83,250,668.84
  20 |   12,328,579.51 |  -70,922,089.34


-70922089.33549142

Formula:

$
\displaystyle NPV = \sum_{t=1}^{t_p} \frac{\mathrm{Cash Flow}_t}{(1 + i)^t}
$  

- $i$ = fixed interest rate  
- $t_p$ = project lifetime  (years)

Outputs:

`plant.pv_array` – present value of each year’s cash flow

`plant.npv_array` – cumulative NPV by year (final entry is project NPV)

---

**Levelized Cost of Product (LCOP)**

Concept: The break-even selling price that sets NPV = 0 over the analysis horizon, given production volumes.

Call:

In [22]:
demo_plant.calculate_levelized_cost(print_results=True)

Levelized cost: 2.127 USD/unit


Computation formula:

$
\displaystyle LCOP = 
\frac{
\sum_{t=1}^{t_p} \dfrac{CAPEX_t + OPEX_t - REV^{\text{side}}_t}{(1 + i)^{t}}
}{
\sum_{t=1}^{t_p} \dfrac{Q_t}{(1 + i)^{t}}
}
$

- $CAPEX_t$ = capital expenditure in year $t$  
- $OPEX_t$ = operating expenditure (cash cost) in year $t$  
- $REV^{\text{side}}_t$ = side product(s) revenue in year $t$, if any (sum of all by-/co-product revenues)  
- $Q_t$ = main-product production output in year $t$  
- $i$ = discount/interest rate  
- $t_p$ = project lifetime (in years)

Outputs:
- `plant.levelized_cost` – scalar levelized cost (for the **main product** after crediting side product revenues)

---

**Payback Time (PBT)**

Concept: First year when cumulative cash flow ≥ 0 (undiscounted payback).

Call:

In [23]:
demo_plant.calculate_payback_time(print_results=True)
demo_plant.calculate_payback_time(print_results=True, additional_capex=True)  # We set additional_capex=True to account for additional CAPEX events in the payback time calculation

Payback time: 6.70 years
Payback time: 6.73 years


Formula:

$
\displaystyle \mathrm{PBT} = \frac{\mathrm{FCI}}{\overline{CF}}
$  
where:  
- $\mathrm{FCI}$ = fixed capital investment  
- $\overline{CF}$ = average annual cash flow  

Outputs:
- `plant.payback_time` – scalar PBT (years)

---

**Return on Investment (ROI)**

Concept: Ratio of average annual profit to total invested capital (definition may vary; this follows your code’s chosen approach).

Call:

In [24]:
demo_plant.calculate_roi(print_results=True)
demo_plant.calculate_roi(print_results=True, additional_capex=True)  # We set additional_capex=True to account for additional CAPEX events in the ROI calculation

Return of investment: 8.06%
Return of investment: 8.04%


Formula:

$
\displaystyle ROI = \frac{\sum_{t=1}^{t_p} \text{Net Profit}_t}{t_p \cdot \sum \text{Total Investment}}
$ 

where:  
- $\text{Net Profit}_t$ = plant net profit in year $t$  
- $\text{Total Investment}$ = invested capital = FCI + working capital

Outputs:
- `plant.roi` – scalar ROI (fraction)

---

**Internal Rate of Return (IRR)**

Concept: Discount rate that makes NPV = 0 for the project cash flows.

Call:

In [25]:
demo_plant.calculate_irr(print_results=True)

Internal Rate of Return: 4.35%


Equation: 

Find IRR such that,
$
\displaystyle 0 = \sum_{t=1}^{t_p} \frac{CF_t}{(1 + IRR)^{t}}
$

Outputs:
- `plant.irr` – scalar IRR (fraction)




---

◀ **Previous:** [Part 1: Defining Equipment](part_1_equipment.ipynb) &nbsp;|&nbsp; **Next:** [Part 3: Cost Analysis and Sensitivity](part_3_analysis.ipynb) ▶